# [Demo] Modelos, Mensagens e Prompts

Primeiro contato prático. Sem framework de agente ainda: só o **modelo**, as **mensagens** e um **prompt**, o suficiente para o Assistente da Clínica Alura responder uma dúvida de paciente.

O que vamos ver:
- utilizar o provedor OpenAI
- inicializar com Langchain o modelo com `ChatOpenAI`;
- conversar com **mensagens** (`SystemMessage`, `HumanMessage`) e ler o `AIMessage` de volta;
- inspecionar o que vem junto da resposta (tokens, motivo de parada);
- reaproveitar instruções com um **prompt template**.

## Setup

Carregamos as variáveis de ambiente (a chave `OPENAI_API_KEY` vem do `.env`) e inicializamos o modelo uma vez.

In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from openai import OpenAI

In [4]:
client = OpenAI()

In [5]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="O que é langchain em uma frase?",
)

print(response.output_text)

LangChain é uma biblioteca para desenvolvimento de aplicações que integram modelos de linguagem, permitindo a construção de pipelines complexos e interações dinâmicas com dados e APIs.


In [6]:
response

Response(id='resp_0986562d1b5bd2d8006a779436e3a08190872970bf4a425f5d', created_at=1786221622.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_0986562d1b5bd2d8006a77943800e88190b8f1c56621c897ca', content=[ResponseOutputText(annotations=[], text='LangChain é uma biblioteca para desenvolvimento de aplicações que integram modelos de linguagem, permitindo a construção de pipelines complexos e interações dinâmicas com dados e APIs.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1786221624.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='in_memory', reasoning=Reasoni

In [7]:
from langchain_openai import ChatOpenAI

In [8]:
# temperature=0 deixa a resposta mais estável (bom para uma demo reproduzível)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1278c74a0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1278f8c20>, root_client=<openai.OpenAI object at 0x107bc4170

In [9]:
type(model)

langchain_openai.chat_models.base.ChatOpenAI

In [10]:
model.invoke("O que é langchain em uma frase?")

AIMessage(content='LangChain é uma biblioteca de código aberto projetada para facilitar a construção de aplicações que utilizam modelos de linguagem, integrando-os com diversas fontes de dados e ferramentas.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 16, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_788bf556eb', 'id': 'chatcmpl-EAi9SoYS7Rjf1FtChjUd3HC1OWxiB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe31c-3ab0-7e33-83c2-a54331b5eb43-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 34, 'total_tokens': 50, 'input_token_details': {

## Mensagens: conversando de forma estruturada

Em vez de mandar uma string solta, a conversa é uma **lista de mensagens** com papéis:

- `SystemMessage`: define o papel e as regras do assistente;
- `HumanMessage`: a fala do usuário (aqui, um paciente).

O modelo responde com um `AIMessage`.

![AI Engineering: usar um LLM e construir uma aplicação](resources/langchain-messages.png)

In [11]:
from langchain.messages import SystemMessage, HumanMessage

In [12]:
messages = [
    SystemMessage("Você é o assistente da Clínica Alura. Responda de forma cordial, curta e objetiva."),
    HumanMessage("Preciso levar meus exames antigos na primeira consulta?"),
]

In [13]:
response = model.invoke(messages)

In [14]:
type(response)

langchain_core.messages.ai.AIMessage

In [15]:
print(response.content)

Sim, é recomendável trazer seus exames antigos na primeira consulta. Isso ajuda o médico a entender melhor seu histórico de saúde.


## A resposta é um objeto rico

`invoke` não devolve só texto: devolve um `AIMessage` com **metadados** úteis, como quantos tokens foram usados e por que o modelo parou. Isso é a base de custo e de observabilidade mais tarde.

In [16]:
print("Tipo:", type(response).__name__)
print("Motivo de parada:", response.response_metadata.get("finish_reason"))

Tipo: AIMessage
Motivo de parada: stop


In [17]:
print("ID de Request da Nuvem:", response.id)
print("Tokens:", response.usage_metadata)

ID de Request da Nuvem: lc_run--019fe31d-7856-77b3-a357-43189929f08b-0
Tokens: {'input_tokens': 42, 'output_tokens': 25, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [18]:
response

AIMessage(content='Sim, é recomendável trazer seus exames antigos na primeira consulta. Isso ajuda o médico a entender melhor seu histórico de saúde.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 42, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_72f05258df', 'id': 'chatcmpl-EAiAlQZDIy6qDB7UqnHe3V5bpoqib', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe31d-7856-77b3-a357-43189929f08b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 25, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'au

## Prompt template: instruções reaproveitáveis

Quando a mesma instrução vale para várias perguntas, um **prompt template** separa a parte fixa (o papel do assistente) da parte variável (a dúvida). Assim não repetimos o `SystemMessage` a cada chamada.

In [19]:
system_prompt = "Você é um assistente útil."
user_input = input("Input: ")

In [20]:
# Usando Python puro com f-strings
print(f"System Prompt: {system_prompt}")
print(f"User Input: {user_input}")

System Prompt: Você é um assistente útil.
User Input: Olá


In [21]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate

In [22]:
template = ChatPromptTemplate.from_messages([
    SystemMessage("Você é o assistente da Clínica Alura. Responda de forma cordial, curta e objetiva."),
    HumanMessagePromptTemplate.from_template("{pergunta}")
])

In [23]:
prompt_value = template.invoke({"pergunta":"Vocês atendem aos sábados?"})
prompt_value.messages

[SystemMessage(content='Você é o assistente da Clínica Alura. Responda de forma cordial, curta e objetiva.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Vocês atendem aos sábados?', additional_kwargs={}, response_metadata={})]

## Classificador simples

O modelo também serve para tarefas pontuais fora de um agente. Aqui pedimos que ele **classifique** a intenção do paciente em uma palavra, o tipo de bloco que dá para reaproveitar num roteamento completo.

In [24]:
classifier = ChatPromptTemplate.from_messages([
    SystemMessage("Classifique a mensagem do paciente em uma única palavra: "
                  "informação, agendamento ou urgência. Responda só a palavra."),
    HumanMessagePromptTemplate.from_template("{mensagem}"),
])

In [25]:
msg = "Estou com uma forte dor no peito agora, o que eu faço?"
print(model.invoke(classifier.invoke({"mensagem": msg})).content)

urgência


In [26]:
msg = "Preciso fazer meu check-up esse mês"
print(model.invoke(classifier.invoke({"mensagem": msg})).content)

agendamento


In [27]:
msg = "Vocês têm alguma unidade em Alphaville?"
print(model.invoke(classifier.invoke({"mensagem": msg})).content)

informação


## Interface Unificada

In [28]:
from langchain.chat_models import init_chat_model

In [29]:
# Inicializa o chat model para modelos diferentes de forma unificada
chat_model = init_chat_model("openai:gpt-4o-mini")

In [30]:
chat_model.invoke("O que é langchain em uma frase?")

AIMessage(content='LangChain é uma biblioteca projetada para facilitar a construção de aplicações que utilizam modelos de linguagem, integrando ferramentas de processamento de linguagem natural e fluxos de trabalho.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 16, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_788bf556eb', 'id': 'chatcmpl-EAiFYbFdujTr5dCkc2We1ovbZDJZl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe322-00d4-7060-a513-e3230f6fe209-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 34, 'total_tokens': 50, 'input_token_det

Explore outros modelos e outras providers. A interface é a mesma. 


```python
>> openai_o3_mini = init_chat_model("openai:o3-mini")

>> claude_sonnet = init_chat_model("claude-sonnet-4-6")

>> gemini_flash = init_chat_model("google_genai:gemini-2.5-flash-lite")
```

## Takeaway

Com **modelo + mensagens + prompt** já dá para responder e até classificar, mas o modelo ainda **não age**.